# Oppgave 4: Varmeligning (1D)
## Problemstilling
Vi skal løse varmeligningen
$$u_t(x,t) = u_{xx}(x,t) - \cos(\pi x), \qquad -1 < x < 1$$
med randbetingelser
$$u(-1,t)=0, \quad u(1,t)=2$$
og initialbetingelse
$$u(x,0)=1+x+5\sin(\pi x).$$


## Diskretisering i rom
Romintervallet diskretiseres med $m=100$ indre punkter.
$$h = \frac{2}{m+1}$$
Andrederiverten approksimeres med sentral differanse:
$$u_{xx}(x_i,t^n) \approx \frac{u_{i-1}^n -2u_i^n + u_{i+1}^n}{h^2}$$

Dette gir systemet
$$\dot{\mathbf{u}} = A\mathbf{u} - F$$
der $A$ er Laplace-matrisen og $F$ er kildetermen med randbidrag.


## Tidsdiskretisering – Euler
Forlengs Euler-metoden brukes:
$$u^{n+1} = u^n + \Delta t(Au^n - F)$$

Stabilitetskravet er
$$r = \frac{\Delta t}{h^2} < \frac{1}{2}$$
Vi velger $r = 0.4$ for å sikre stabilitet.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Romgitter ---
m = 100
x = np.linspace(-1, 1, m + 2)
h = x[1] - x[0]

# --- Laplace-matrise ---
L = (1 / h**2) * (
    np.diag((m - 1) * [1], -1) +
    np.diag(m * [-2], 0) +
    np.diag((m - 1) * [1], 1)
)

# --- Kildeledd og randbidrag ---
# Dirichletbetingelser: u(-1) = a = 0,  u(1) = B = 2
a = 0
B = 2
f = np.cos(np.pi * x[1:-1])
F = f.copy()
F[0]  -= a / h**2   # randbidrag venstre  (a=0, ingen effekt her)
F[-1] -= B / h**2   # randbidrag høyre

# --- Tidsparametere ---
r  = 0.4            # Courant-tall (< 0.5 for stabilitet)
dt = r * h**2
T  = 2.0
N  = int(T / dt) + 1

print(f"h  = {h:.5f}")
print(f"dt = {dt:.6f}")
print(f"r  = {dt/h**2:.2f}")
print(f"N  = {N} tidssteg")


## Euler-integrasjon
Vi implementerer Euler-metoden som en funksjon og lagrer løsningen i utvalgte tidspunkter for plotting.


In [ ]:
def euler(g, x0, t0, t1, N):
    """Forlengs Euler-metode for u' = g(u, t)."""
    t   = np.linspace(t0, t1, N)
    dt  = t[1] - t[0]
    out = np.zeros((N, x0.size))
    out[0, :] = x0
    for n in range(N - 1):
        out[n + 1, :] = out[n, :] + dt * g(out[n, :], t[n])
    return out, t

# --- Høyreside for ODE: u' = A*u - F ---
A = L
def g(u, t):
    return A @ u - F

# --- Initialbetingelse ---
u0 = 1.0 + x[1:-1] + 5 * np.sin(np.pi * x[1:-1])

# --- Kjør Euler ---
u_varme, t = euler(g, u0, 0, T, N)


## Stasjonær løsning
Når $t \to \infty$ blir $u_t = 0$ og ligningen reduseres til Poissonligningen
$$u_{xx} = \cos(\pi x)$$
med samme randbetingelser som i Oppgave 3. Den analytiske stasjonære løsningen er derfor
$$u_{\infty}(x) = -\frac{\cos(\pi x)}{\pi^2} + x + 1 - \frac{1}{\pi^2}.$$


In [ ]:
def u_stasjonaer(x):
    return -np.cos(np.pi * x) / np.pi**2 + x + 1 - 1 / np.pi**2

# --- Indekser for fire tidspunkter ---
idx1 = N // 20
idx2 = N // 4
idx3 = N - 1

# --- Plot: tidsutvikling + stasjonær løsning ---
x_fine = np.linspace(-1, 1, 200)

plt.figure(figsize=(8, 5))
plt.plot(x[1:-1], u_varme[0,    :], label=f"t = {t[0]:.2f}  (initialbetingelse)")
plt.plot(x[1:-1], u_varme[idx1, :], label=f"t = {t[idx1]:.2f}")
plt.plot(x[1:-1], u_varme[idx2, :], label=f"t = {t[idx2]:.2f}")
plt.plot(x[1:-1], u_varme[idx3, :], label=f"t = {t[idx3]:.2f}  (sluttid)")
plt.plot(x,       u_stasjonaer(x),  'k--', linewidth=2, label="Stasjonær løsning (oppg. 3)")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title(r"Varmeligning: $u_t = u_{xx} - \cos(\pi x)$")
plt.legend()
plt.grid(True)
plt.show()

# --- Maks avvik fra stasjonær løsning ved t = T ---
feil = np.max(np.abs(u_varme[-1, :] - u_stasjonaer(x[1:-1])))
print(f"Maks avvik fra stasjonær løsning ved t={T}: {feil:.2e}")
